# Hard Case: WebDriverWait (waiting for elements)

On dynamic websites, elements often **don't exist yet** when a page first opens (JavaScript is still
running). If you call `find_element` right away, you get an empty result or an error.

The **bad** beginner solution: `time.sleep(5)` — it wastes time when the element is ready from
the first second, and still fails if it actually takes 6 seconds.

The **right** solution: **`WebDriverWait`** + **`expected_conditions`** — wait **until the
condition is met** (up to some number of seconds), then continue as soon as it's ready. Adaptive and efficient.

We'll practice on `https://quotes.toscrape.com/js/` (the quotes are rendered with JavaScript).

**Tooling:** `selenium` (`WebDriverWait`, `expected_conditions`).


## Example 1 — Waiting for an element to appear

`WebDriverWait(driver, 10).until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, ".quote")))`
means: "wait up to 10 seconds until `.quote` elements exist on the page, then return
those elements." If they still aren't there after 10 seconds → `TimeoutException`.


In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC


def make_driver(headless=True):
    o = Options()
    if headless:  # set False if you want to watch the browser
        o.add_argument("--headless=new")
    o.add_argument("--window-size=1280,900")
    return webdriver.Chrome(options=o)


driver = make_driver()
try:
    driver.get("https://quotes.toscrape.com/js/")

    # wait until the .quote elements (the JS render result) appear
    quotes = WebDriverWait(driver, 10).until(
        EC.presence_of_all_elements_located((By.CSS_SELECTOR, ".quote"))
    )
    print("Number of quotes after waiting:", len(quotes))
    print("Example:", quotes[0].find_element(By.CSS_SELECTOR, ".text").text)
finally:
    driver.quit()


Number of quotes after waiting: 10
Example: “The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”


## Example 2 — Wait until an element is clickable, then click it

`expected_conditions` offers many conditions. The most commonly used ones:

| Condition | Meaning |
| --- | --- |
| `presence_of_element_located` | the element exists in the DOM |
| `visibility_of_element_located` | the element exists **and** is visible |
| `element_to_be_clickable` | the element is ready to be clicked (visible & enabled) |
| `text_to_be_present_in_element` | specific text has appeared |
| `title_contains` | the page title contains some text |

Below: wait until the Next button **is clickable** → click → wait for the page 2 content to render.


In [2]:
driver = make_driver()
try:
    driver.get("https://quotes.toscrape.com/js/")
    wait = WebDriverWait(driver, 10)

    # 1) make sure the page 1 quotes exist
    wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, ".quote")))
    quote_page1 = driver.find_element(By.CSS_SELECTOR, ".quote .text").text

    # 2) wait until the Next button IS CLICKABLE, then click it
    next_btn = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "li.next a")))
    next_btn.click()

    # 3) wait for the page 2 content to render (URL changes to /page/2/)
    wait.until(EC.url_contains("/page/2"))
    wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, ".quote")))
    quote_page2 = driver.find_element(By.CSS_SELECTOR, ".quote .text").text

    print("Current URL  :", driver.current_url)
    print("Quote page 1 :", quote_page1[:50], "...")
    print("Quote page 2 :", quote_page2[:50], "...")
finally:
    driver.quit()


Current URL  : https://quotes.toscrape.com/js/page/2/
Quote page 1 : “The world as we have created it is a process of o ...
Quote page 2 : “This life is what you make it. No matter what, yo ...


## Conclusion & Exercise

- `WebDriverWait` + `expected_conditions` = waiting **smartly**, not a blind `time.sleep`.
- Wrap common patterns into helper functions so you can reuse them.
- Handle `TimeoutException` so the program doesn't crash when an element truly never appears.

**Exercise:** write a helper `get_text(driver, css, timeout=10)` that waits for an element to appear
and then returns its text (or `None` on timeout). A sample solution is in the next cell.


In [3]:
# Sample solution to the exercise
from selenium.common.exceptions import TimeoutException


def get_text(driver, css, timeout=10):
    try:
        el = WebDriverWait(driver, timeout).until(
            EC.visibility_of_element_located((By.CSS_SELECTOR, css))
        )
        return el.text
    except TimeoutException:
        return None  # element did not appear within the given time


driver = make_driver()
try:
    driver.get("https://quotes.toscrape.com/js/")
    print("Found     :", get_text(driver, ".quote .text"))
    print("Not found :", get_text(driver, ".nonexistent-element", timeout=3))
finally:
    driver.quit()


Found     : “The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”


Not found : None
